# Ejercicio 2. Anonimización de datos en reposo

Curso CIB-209, Temas Especiales en Seguridad de Datos y Sistemas.

Modificar únicamente la celda CONFIGURACIÓN. Ejecutar después todas las celdas en orden.

## Recordatorio de la técnica

Este ejercicio no usa aprendizaje automático. Usa tres técnicas de protección de datos personales y luego dos ataques contra el resultado.

SHA-256 es una función de resumen, no un cifrado. Convierte la cédula en una cadena fija y no existe una operación que la devuelva al valor original. Eso no significa que sea irreversible en la práctica: si el atacante conoce la lista de cédulas posibles, puede calcular el resumen de todas y comparar. Ese es el ataque de diccionario. La sal es un valor secreto que se agrega antes de calcular el resumen, de modo que quien no la conoce no puede reproducirlo.

La generalización no oculta el dato, le baja el detalle: la dirección IP pasa de identificar el equipo a identificar solo la red, y la fecha pasa de identificar el momento a identificar el mes. La supresión elimina la columna por completo.

## Vocabulario

- Resumen criptográfico: salida de longitud fija que representa un valor de entrada.
- Sal: valor secreto que se agrega antes de calcular el resumen.
- Seudonimización: se sustituye el identificador por un código. Sigue siendo dato personal.
- Anonimización: no queda forma razonable de volver a la persona.
- Reidentificación: volver a asociar un registro protegido con una persona concreta.
- Fuente auxiliar: información que el atacante ya tiene, en este caso la tabla de personal.

## Cómo se leen las salidas

Este ejercicio no produce gráficos, produce tablas. Hay tres tipos de salida:

1. La tabla protegida, para ver qué columnas sobrevivieron y con cuánto detalle.
2. El estado de las tres consultas de investigación, con tres valores posibles: funciona, parcial o falla. Parcial significa que la consulta corre pero ya no da el detalle que una investigación necesita.
3. El resultado de los dos ataques, en cantidad de personas reidentificadas sobre 40, y el tiempo que tardó el ataque de diccionario.

Este recordatorio explica la técnica y cómo leer las salidas. No interpreta los resultados, eso le corresponde al grupo.

In [ ]:
# ======================= CONFIGURACION =======================
NUMERO_DE_GRUPO = "G00"

# opciones: "seudonimizacion_sin_sal", "seudonimizacion_con_sal", "generalizacion"
metodo_de_proteccion = "seudonimizacion_sin_sal"

precision_de_ip = "completa"     # opciones: "completa", "segmento", "suprimida"
precision_de_fecha = "exacta"    # opciones: "exacta", "dia", "mes"
# =============================================================

In [ ]:
import hashlib
import time
import pandas as pd

def sello_de_corrida(**parametros):
    texto = "|".join(f"{k}={parametros[k]}" for k in sorted(parametros))
    return hashlib.md5(texto.encode()).hexdigest()[:4].upper()

SAL = "cib209-sal-fija-2026"
eventos = pd.read_csv("eventos_seguridad.csv")
personal = pd.read_csv("empleados_rrhh.csv")

print("Grupo:", NUMERO_DE_GRUPO)
print("metodo_de_proteccion =", metodo_de_proteccion)
print("precision_de_ip =", precision_de_ip, "  precision_de_fecha =", precision_de_fecha)
print("Sello de la corrida:", sello_de_corrida(metodo=metodo_de_proteccion, ip=precision_de_ip, fecha=precision_de_fecha))
print()
print("Eventos cargados:", len(eventos), " Personas en la tabla de personal:", len(personal))
print("Columnas del origen:", ", ".join(eventos.columns))

In [ ]:
protegida = eventos.copy()
if metodo_de_proteccion == "seudonimizacion_sin_sal":
    protegida["id_persona"] = [hashlib.sha256(c.encode()).hexdigest()[:16] for c in protegida.cedula]
elif metodo_de_proteccion == "seudonimizacion_con_sal":
    protegida["id_persona"] = [hashlib.sha256((SAL + c).encode()).hexdigest()[:16] for c in protegida.cedula]
protegida = protegida.drop(columns=["cedula", "correo"])

if precision_de_ip == "segmento":
    protegida["direccion_ip"] = protegida.direccion_ip.str.rsplit(".", n=1).str[0] + ".0/24"
elif precision_de_ip == "suprimida":
    protegida["direccion_ip"] = "suprimida"

if precision_de_fecha == "mes":
    protegida["fecha"] = protegida.fecha.str[:7]
    protegida = protegida.drop(columns=["hora"])
elif precision_de_fecha == "dia":
    protegida = protegida.drop(columns=["hora"])

print("TABLA PROTEGIDA, PRIMERAS OCHO FILAS")
print(protegida.head(8).to_string(index=False))
print()
print("Columnas que quedan:", ", ".join(protegida.columns))

In [ ]:
print("CONSULTA 1. Reconstruir la secuencia de accesos de una persona")
if "id_persona" not in protegida.columns:
    estado_1 = "falla"
    print("No existe identificador de persona en la tabla protegida.")
else:
    objetivo = protegida.id_persona.value_counts().index[0]
    sec = protegida[protegida.id_persona == objetivo]
    orden = ["fecha", "hora"] if "hora" in protegida.columns else ["fecha"]
    estado_1 = "funciona" if "hora" in protegida.columns else "parcial"
    print("Persona seudonimizada:", objetivo, " eventos:", len(sec))
    print(sec.sort_values(orden).head(6).to_string(index=False))
print("Estado de la consulta 1:", estado_1)

In [ ]:
print("CONSULTA 2. Accesos fuera de horario por área")
tabla_2 = protegida[protegida.fuera_de_horario == 1].area.value_counts().rename("accesos_fuera_de_horario")
estado_2 = "funciona"
print(tabla_2.to_string())
print("Estado de la consulta 2:", estado_2)
print()
print("CONSULTA 3. Localizar el equipo desde el que se hizo el acceso")
valor_ip = str(protegida.direccion_ip.iloc[0])
if valor_ip == "suprimida":
    estado_3 = "falla"
elif "/24" in valor_ip:
    estado_3 = "parcial"
else:
    estado_3 = "funciona"
print(protegida.direccion_ip.value_counts().head(6).to_string())
print("Estado de la consulta 3:", estado_3)

In [ ]:
print("ATAQUE 1. Diccionario sobre el identificador seudonimizado")
if "id_persona" not in protegida.columns:
    reidentificados_diccionario = 0
    segundos = 0.0
    print("La tabla no tiene identificador de persona, el ataque no aplica.")
else:
    inicio = time.perf_counter()
    diccionario = {hashlib.sha256(c.encode()).hexdigest()[:16]: c for c in personal.cedula}
    encontrados = set(protegida.id_persona.unique()) & set(diccionario)
    segundos = time.perf_counter() - inicio
    reidentificados_diccionario = len(encontrados)
    print("Cédulas probadas:", len(personal))
    print("Coincidencias encontradas:", reidentificados_diccionario)
print("Personas reidentificadas por diccionario:", reidentificados_diccionario, "de", len(personal))
print("Tiempo del ataque en segundos:", round(segundos, 4))

In [ ]:
print("ATAQUE 2. Cruce con la tabla de personal usando la dirección de red")
if valor_ip == "suprimida":
    reidentificados_cruce = 0
    muestra = pd.DataFrame()
else:
    if "/24" in valor_ip:
        llave = personal.ip_equipo_asignado.str.rsplit(".", n=1).str[0] + ".0/24"
    else:
        llave = personal.ip_equipo_asignado
    tabla = personal.assign(llave=llave)
    conteo = tabla.llave.value_counts()
    unicas = set(conteo[conteo == 1].index)
    presentes = set(protegida.direccion_ip.unique())
    identificadas = tabla[tabla.llave.isin(unicas) & tabla.llave.isin(presentes)]
    reidentificados_cruce = len(identificadas)
    muestra = identificadas[["nombre_completo", "area", "puesto", "llave"]].head(6)
print("Personas reidentificadas por cruce:", reidentificados_cruce, "de", len(personal))
if len(muestra):
    print()
    print("Muestra de personas identificadas con nombre y apellido")
    print(muestra.to_string(index=False))

In [ ]:
print("RESUMEN DE LA CORRIDA")
print("Grupo:", NUMERO_DE_GRUPO, " Sello:", sello_de_corrida(metodo=metodo_de_proteccion, ip=precision_de_ip, fecha=precision_de_fecha))
resumen = pd.DataFrame([
    ["metodo_de_proteccion", metodo_de_proteccion],
    ["precision_de_ip", precision_de_ip],
    ["precision_de_fecha", precision_de_fecha],
    ["consulta 1, secuencia de una persona", estado_1],
    ["consulta 2, fuera de horario por área", estado_2],
    ["consulta 3, localizar el equipo", estado_3],
    ["personas reidentificadas por diccionario", f"{reidentificados_diccionario} de {len(personal)}"],
    ["personas reidentificadas por cruce", f"{reidentificados_cruce} de {len(personal)}"],
    ["segundos del ataque de diccionario", round(segundos, 4)],
], columns=["concepto", "valor"])
print(resumen.to_string(index=False))